# DLR Interchange


Models the London contactless network as a graph to find the most likely interchange points between DLR and other services based on shortest journey time.

**Idea:** Stations and line combinations are nodes; travel times, interchange times and wait times are edges.


**Output:** A mapping table of all DLR-to-non-DLR journeys with optimal in-station interchange points.

In [0]:
import pandas as pd
import json
import time
import urllib.parse
import urllib.request
import requests
import re
import os

# %pip install networkx
import networkx as px_net

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

print("Libraries installed")

try:
  TFL_API_KEY = dbutils.secrets.get(
      scope="payments-secrets", key="tfl-primary-key"
  )
  print("TfL API key found")
except Exception:
  print("Warning: TfL API key not found")

Libraries installed
TfL API key found


### Network Nodes: All Contactless Stations

As of November 2026, there are currently around 860 contactless rail stations. This number is expected to grow as Project Oval expands the contactless network.

Each node in the network represents a specific station-line pair. <br>
This analysis relies on the TfL Journey Planner API, which can be queried using ICS and NaPTAN station codes.

The `all_stations` reference table is based on `ref_nlc` and was manually reviewed against the MCaaS Portal.

Bus, tram, and river services are excluded.

##### Station Code Definitions

**NLC (National Location Code)**: <br>
Represents a set of physical ticket gates. It separates multi-modal stations (for example, London Bridge National Rail and London Underground) into distinct graph nodes.

**ICS (Infrastructure Code System)**: <br>
TfL's station-level identifier used for Journey Planner API lookups.

**NaPTAN (National Public Transport Access Node)**: <br>
A national standard identifier used across the UK transport network to provide precise station-level granularity. Codes are typically prefixed with 940GZZLU for London Underground stations, 940GZZDL for DLR stations, and 910G or 490G for National Rail stations.

In [0]:
## loading a new station reference table from a csv
# station_reference = pd.read_csv("/Workspace/Payments/AnnaIvahnenko@tfl.gov.uk/all_stations_ref.csv")
# station_reference = spark.createDataFrame(station_reference)

# # Save as a reference table
# (
#     station_reference.write
#       .mode("overwrite")
#       .format("delta")
#       .saveAsTable(
#           "adlsg2analyticsprd001_sandbox_payments.ref.station_reference"
#       )
# )

# # check that it is saved
# display(
#     spark.table(
#         "adlsg2analyticsprd001_sandbox_payments.ref.station_reference"
#     )
# )

station_reference = spark.table("adlsg2analyticsprd001_sandbox_payments.ref.station_reference")
display(station_reference)

Station,Line,ICS,NaPTAN,NLC,Gate_NLC
Abbey Road,dlr,1003006,940GZZDLABR,750,750
Abbey Wood,elizabeth,1001001,9100ABWD0,5131,5131
Abbey Wood,thameslink,1001001,9100ABWD0,5131,5131
Abbey Wood,southeastern,1001001,910GABWD,5131,5131
Abbey Wood,thameslink,1001001,910GABWD,5131,5131
Abbey Wood,elizabeth,1001001,910GABWDXR,5131,5131
Acton Central,mildmay,1001002,910GACTNCTL,1404,1404
Acton Main Line,elizabeth,1001003,9100ACTONML0,3000,3000
Acton Main Line,great-western-railway,1001003,910GACTONML,3000,3000
Acton Town,district,1000002,940GZZLUACT,500,500


**To do**: Check whether there are any stations in the ref_nlc table that are not present in this reference table. This is difficult because station names are not always consistent, and there are nuances in how NLC codes are used within MCaaS.

## Network Edges
The next cells gather the three edge-weight inputs: **travel times**, **wait times**, and **line interchange times**.

### Adjacent Station Travel Times

**Source:** Derived from TfL Line Route Sequence feeds and queried via the TfL Journey Planner API for all rail modes (Tube, DLR, Overground, Elizabeth Line and National Rail).

**Assumptions & Fallbacks:**
- Only adjacent station pairs along active branch sequences are queried.
- If a live API query fails or returns zero duration, a fallback travel time of 3 minutes is used (avg. travel time between stations) 

In [0]:
# ==============================================================================
# TRAVEL TIMES BUILDER
# ==============================================================================

RAIL_MODES = ["tube", "dlr", "overground", "elizabeth-line", "national-rail"]
DEFAULT_TRAVEL_TIME = 3.0


def normalize_line(name):
  if not name:
    return ""
  return (
      str(name)
      .lower()
      .replace(" line", "")
      .replace("-", "")
      .replace("_", "")
      .strip()
  )


# 2. Load reference table from Databricks Spark catalog
ref_df = spark.table("adlsg2analyticsprd001_sandbox_payments.ref.station_reference").toPandas()

# Exclude redundant operators
ref_df = ref_df[~ref_df["Line"].isin(["tfl-rail", "london-overground"])].copy()

# Construct Node_ID if not already present
if "Node_ID" not in ref_df.columns:
  ref_df["Node_ID"] = (
      ref_df["Station"].astype(str)
      + "_"
      + ref_df["Line"].astype(str)
      + "_"
      + ref_df["Gate_NLC"].astype(str)
  )

# Build lookup dictionaries: (norm_line, naptan) -> Node_ID and (norm_line, ics) -> Node_ID
naptan_lookup = {}
ics_lookup = {}
valid_nodes = set()

for _, row in ref_df.iterrows():
  node_id = str(row["Node_ID"]).strip()
  norm_line = normalize_line(row["Line"])
  naptan = str(row["NaPTAN"]).strip()
  ics = str(row["ICS"]).strip()

  if pd.notna(node_id) and node_id != "nan":
    valid_nodes.add(node_id)
    if pd.notna(naptan) and naptan != "nan":
      naptan_lookup[(norm_line, naptan)] = node_id
    if pd.notna(ics) and ics != "nan" and ics != "0":
      ics_lookup[(norm_line, ics)] = node_id

print(
    f"Loaded {len(valid_nodes):,} valid nodes with NaPTAN and ICS fallback"
    " mappings."
)

# 3. Query TfL API for routes and travel times
params = {"app_key": TFL_API_KEY} if TFL_API_KEY else {}
lines_url = f"https://api.tfl.gov.uk/Line/Mode/{','.join(RAIL_MODES)}"

res = requests.get(lines_url, params=params)
if res.status_code != 200:
  raise Exception(f"Failed to fetch lines from TfL API: {res.status_code}")

lines = res.json()
records = []

print(f"Querying adjacent stop pairs across {len(lines)} lines...")

for line in lines:
  line_id = line["id"].lower()
  line_name = line["name"]
  norm_line = normalize_line(line_name)

  seq_url = f"https://api.tfl.gov.uk/Line/{line_id}/Route/Sequence/all"
  seq_res = requests.get(seq_url, params=params)
  if seq_res.status_code != 200:
    continue

  seq_data = seq_res.json()

  for branch in seq_data.get("stopPointSequences", []):
    stops = branch.get("stopPoint", [])

    for i in range(len(stops) - 1):
      from_stop, to_stop = stops[i], stops[i + 1]

      # Extract NaPTAN and potential ICS identifiers from TfL API stop payload
      from_id = from_stop.get("id", "")
      to_id = to_stop.get("id", "")
      from_name = from_stop.get("name", "")
      to_name = to_stop.get("name", "")

      # Try NaPTAN first; if missing/unmatched, fallback to ICS if available in stop properties
      from_ics = str(from_stop.get("icsCode", ""))
      to_ics = str(to_stop.get("icsCode", ""))

      from_node_id = naptan_lookup.get(
          (norm_line, from_id),
          ics_lookup.get(
              (norm_line, from_ics), naptan_lookup.get(from_id, from_id)
          ),
      )
      to_node_id = naptan_lookup.get(
          (norm_line, to_id),
          ics_lookup.get((norm_line, to_ics), naptan_lookup.get(to_id, to_id)),
      )

      # Scope check: Both nodes must exist in reference table
      if from_node_id not in valid_nodes or to_node_id not in valid_nodes:
        continue

      duration = None
      status = "Success"

      # Query Journey Planner API for running time
      jp_url = f"https://api.tfl.gov.uk/Journey/JourneyResults/{urllib.parse.quote(from_id)}/to/{urllib.parse.quote(to_id)}"

      try:
        jp_res = requests.get(jp_url, params=params, timeout=4)
        if jp_res.status_code == 200:
          journeys = jp_res.json().get("journeys", [])
          if journeys:
            legs = journeys[0].get("legs", [])
            if legs:
              raw_dur = legs[0].get("duration", journeys[0].get("duration"))
              if raw_dur is not None:
                duration = float(raw_dur)
      except Exception:
        pass

      if duration is None or duration <= 0:
        duration = DEFAULT_TRAVEL_TIME
        status = "Fallback Used"

      # Forward edge
      records.append({
          "Line": line_name,
          "From_Node_ID": from_node_id,
          "From_NaPTAN": from_id,
          "From_Station": from_name,
          "To_Node_ID": to_node_id,
          "To_NaPTAN": to_id,
          "To_Station": to_name,
          "Travel_Time_Mins": duration,
          "Status": status,
      })

      # Symmetric Reverse edge
      records.append({
          "Line": line_name,
          "From_Node_ID": to_node_id,
          "From_NaPTAN": to_id,
          "From_Station": to_name,
          "To_Node_ID": from_node_id,
          "To_NaPTAN": from_id,
          "To_Station": from_name,
          "Travel_Time_Mins": duration,
          "Status": f"{status} (Symmetric)",
      })

      time.sleep(0.01)

# 4. Convert results into a Spark DataFrame and save to Delta table
pdf = pd.DataFrame(records).drop_duplicates(
    subset=["Line", "From_Node_ID", "To_Node_ID"]
)

adjacent_station_travel_times = spark.createDataFrame(pdf)
adjacent_station_travel_times.write.mode("overwrite").saveAsTable("adlsg2analyticsprd001_sandbox_payments.ref.adjacent_station_travel_times")

print(
    f"Successfully generated and saved {adjacent_station_travel_times.count():,} travel time edges"
)

Loaded 1,242 valid nodes with NaPTAN and ICS fallback mappings.
Querying adjacent stop pairs across 44 lines...
Successfully generated and saved 1,812 travel time edges


In [0]:
display(adjacent_station_travel_times)

Line,From_Node_ID,From_NaPTAN,From_Station,To_Node_ID,To_NaPTAN,To_Station,Travel_Time_Mins,Status
Bakerloo,Harrow & Wealdstone_bakerloo_597,940GZZLUHAW,Harrow & Wealdstone Underground Station,Kenton_bakerloo_620,940GZZLUKEN,Kenton Underground Station,2.0,Success
Bakerloo,Kenton_bakerloo_620,940GZZLUKEN,Kenton Underground Station,Harrow & Wealdstone_bakerloo_597,940GZZLUHAW,Harrow & Wealdstone Underground Station,2.0,Success (Symmetric)
Bakerloo,Kenton_bakerloo_620,940GZZLUKEN,Kenton Underground Station,South Kenton_bakerloo_709,940GZZLUSKT,South Kenton Underground Station,2.0,Success
Bakerloo,South Kenton_bakerloo_709,940GZZLUSKT,South Kenton Underground Station,Kenton_bakerloo_620,940GZZLUKEN,Kenton Underground Station,2.0,Success (Symmetric)
Bakerloo,South Kenton_bakerloo_709,940GZZLUSKT,South Kenton Underground Station,North Wembley_bakerloo_659,940GZZLUNWY,North Wembley Underground Station,2.0,Success
Bakerloo,North Wembley_bakerloo_659,940GZZLUNWY,North Wembley Underground Station,South Kenton_bakerloo_709,940GZZLUSKT,South Kenton Underground Station,2.0,Success (Symmetric)
Bakerloo,North Wembley_bakerloo_659,940GZZLUNWY,North Wembley Underground Station,Wembley Central_bakerloo_751,940GZZLUWYC,Wembley Central Underground Station,2.0,Success
Bakerloo,Wembley Central_bakerloo_751,940GZZLUWYC,Wembley Central Underground Station,North Wembley_bakerloo_659,940GZZLUNWY,North Wembley Underground Station,2.0,Success (Symmetric)
Bakerloo,Wembley Central_bakerloo_751,940GZZLUWYC,Wembley Central Underground Station,Stonebridge Park_bakerloo_717,940GZZLUSGP,Stonebridge Park Underground Station,3.0,Success
Bakerloo,Stonebridge Park_bakerloo_717,940GZZLUSGP,Stonebridge Park Underground Station,Wembley Central_bakerloo_751,940GZZLUWYC,Wembley Central Underground Station,3.0,Success (Symmetric)


### Network Edges: Expected Line Wait Times

**Source:** Extracted dynamically using the TfL Timetable API (`/Line/{id}/Timetable/{stop}`) at designated benchmark hub stations (`LINE_BENCHMARK_STATIONS`).

**Assumptions & Fallbacks:**
* Passenger arrivals are assumed to be uniform and random.
* Expected passenger wait time is computed using the half-headway rule: $\\text{Wait} = \\frac{60}{2 \\times \\text{TPH}}$ (Trains Per Hour), measuring peak arrivals at hour 08:00–09:00 and off-peak at 11:00–12:00.
* Where timetable feeds are unavailable or fail, rule-based mode-level headways are applied as explicit fallbacks (e.g., Tube/DLR: 2.5m, Elizabeth Line: 2.0m, Overground: 5.0m, National Rail: 7.5m).

In [0]:
# ==============================================================================
#  LINE WAIT TIMES
# ==============================================================================

# Benchmark hub stations used to calculate line frequencies (TPH)
LINE_BENCHMARK_STATIONS = {
    "victoria": "940GZZLUOXC",  # Oxford Circus
    "jubilee": "940GZZLUBST",  # Baker Street
    "northern": "940GZZLUBNK",  # Bank
    "central": "940GZZLUOXC",  # Oxford Circus
    "bakerloo": "940GZZLUBST",  # Baker Street
    "piccadilly": "940GZZLUGPK",  # Green Park
    "district": "940GZZLUWMS",  # Westminster
    "circle": "940GZZLUBST",  # Baker Street
    "hammersmith-city": "940GZZLUBST",  # Baker Street
    "metropolitan": "940GZZLUBST",  # Baker Street
    "waterloo-city": "940GZZLUWLO",  # Waterloo
    "dlr": "940GZZDLWHM",  # West Ham
    "elizabeth": "940GZZLUTCR",  # Tottenham Court Road
}


def calculate_wait_from_tph(trains_per_hour: float) -> float:
  """Computes expected passenger wait time using half-headway: Wait = 60 / (2 * TPH)."""
  if trains_per_hour <= 0:
    return 10.0
  return round(60.0 / (2.0 * trains_per_hour), 1)


def fetch_default_wait_from_api(line_id: str, stop_naptan: str) -> float:
  """Queries TfL Timetable API to calculate default average wait time."""
  params = {"app_key": TFL_API_KEY} if TFL_API_KEY else {}
  url = f"https://api.tfl.gov.uk/Line/{line_id}/Timetable/{stop_naptan}"

  try:
    res = requests.get(url, params=params, timeout=5)
    if res.status_code == 200:
      data = res.json()
      routes = data.get("timetable", {}).get("routes", [])
      peak_count = 0
      off_peak_count = 0

      for route in routes:
        for schedule in route.get("schedules", []):
          for entry in schedule.get("knownJourneys", []):
            hour = int(entry.get("hour", 0))
            if hour == 8:
              peak_count += 1
            elif hour == 11:
              off_peak_count += 1

      if peak_count > 0:
        tph_peak = max(2.0, float(peak_count))
        tph_offpeak = max(1.0, float(off_peak_count))
        return calculate_wait_from_tph((tph_peak + tph_offpeak) / 2.0)
  except Exception:
    pass

  return None


def refresh_wait_times():
  params = {"app_key": TFL_API_KEY} if TFL_API_KEY else {}
  modes = ["tube", "dlr", "overground", "elizabeth-line", "national-rail"]
  url = f"https://api.tfl.gov.uk/Line/Mode/{','.join(modes)}"

  rows = []
  res = requests.get(url, params=params)

  if res.status_code == 200:
    lines_data = res.json()

    for item in lines_data:
      line_name = item.get("name", "").strip()
      mode = item.get("modeName", "").strip()
      line_id = item.get("id", "").strip().lower()

      # 1. Try dynamic API Timetable extraction
      naptan = LINE_BENCHMARK_STATIONS.get(line_id)
      default_w = (
          fetch_default_wait_from_api(line_id, naptan) if naptan else None
      )

      # 2. Rule-based fallbacks if API fails or no benchmark station exists
      if default_w is None:
        if mode == "tube":
          default_w = 2.5
        elif mode == "dlr":
          default_w = 2.5
        elif mode == "elizabeth-line":
          default_w = 2.0
        elif mode == "overground":
          default_w = 5.0
        else:  # National Rail
          default_w = 7.5

      rows.append(
          (line_name, mode, default_w)
      )  # Tuple matching schema order

  # Define explicit PySpark schema
  schema = StructType([
      StructField("Line", StringType(), True),
      StructField("Mode", StringType(), True),
      StructField("Wait", DoubleType(), True),
  ])

  # Create PySpark DataFrame from Python list
  spark_df = spark.createDataFrame(rows, schema)
  spark_df = spark_df.orderBy("Mode", "Line")

  # Write to table
  spark_df.write.mode("overwrite").saveAsTable("adlsg2analyticsprd001_sandbox_payments.ref.line_wait_times")
  print(
      f"Successfully generated and saved {spark_df.count():,} line wait time"
      " profiles to Delta table 'line_wait_times'."
  )
  return spark_df


wait_times = refresh_wait_times()
display(wait_times)

Successfully generated and saved 44 line wait time profiles to Delta table 'line_wait_times'.


Line,Mode,Wait
DLR,dlr,2.5
Elizabeth line,elizabeth-line,2.0
Avanti West Coast,national-rail,7.5
Chiltern Railways,national-rail,7.5
CrossCountry,national-rail,7.5
East Midlands Railway,national-rail,7.5
Gatwick Express,national-rail,7.5
Grand Central,national-rail,7.5
Great Northern,national-rail,7.5
Great Western Railway,national-rail,7.5


### Network Edges: Interchange Walk Times

**Source:** Jorney Planner API.

**Assumptions & Fallbacks:**
* **In-Station Interchanges:** Evaluated using a 4-tier hierarchy:
    1. *Empirical Layout Overrides* (e.g. Bank, King's Cross, Waterloo complex hubs).
    2. *Cross-Platform Benchmarks* (2.0m flat penalty for same-platform transfers).
    3. *Synthetic API Triads* (Walk = Journey A to C - (Leg_A + Leg_B + Wait_B)).
    4. *Depth-Tier Heuristics* (Fallback rule based on surface, subsurface, and deep tube layering).

Only behing the gate interchanges are considered in this analysis, as a customer would need to tap in and out otherwise.

In [0]:
# ==============================================================================
#  BEHIND-THE-BARRIER INTERCHANGE
# ==============================================================================

DEFAULT_INTERCHANGE_TIME = 2.0

# 2. Load station reference table from Databricks Spark catalog
ref_df = spark.table("adlsg2analyticsprd001_sandbox_payments.ref.station_reference").toPandas()

# Exclude redundant operators
ref_df = ref_df[~ref_df["Line"].isin(["tfl-rail", "london-overground"])].copy()

# Ensure Node_ID exists
if "Node_ID" not in ref_df.columns:
  ref_df["Node_ID"] = (
      ref_df["Station"].astype(str) + "_" + 
      ref_df["Line"].astype(str) + "_" + 
      ref_df["Gate_NLC"].astype(str)
  )

# Helper function to parse single or comma-separated Gate_NLC entries
def parse_gate_nlcs(val):
  gate_set = set()
  if pd.notna(val):
    parts = str(val).split(",")
    for p in parts:
      p_clean = p.strip()
      if p_clean.isdigit():
        gate_set.add(int(p_clean))
  return gate_set

# Map each Node_ID to its station attributes and set of Gate_NLC integers
node_meta = {}
for _, row in ref_df.iterrows():
  node_id = str(row["Node_ID"]).strip()
  if pd.notna(node_id) and node_id != "nan":
    node_meta[node_id] = {
        "Station": row["Station"],
        "Line": row["Line"],
        "NaPTAN": str(row["NaPTAN"]).strip(),
        "Gate_NLCs": parse_gate_nlcs(row["Gate_NLC"])
    }

nodes = list(node_meta.keys())
interchanges = []
params = {"app_key": TFL_API_KEY} if TFL_API_KEY else {}

print(f"Evaluating behind-the-barrier interchanges across {len(nodes)} nodes...")

# 3. Compare all unique node pairs sharing at least one common Gate_NLC
for i in range(len(nodes)):
  for j in range(i + 1, len(nodes)):
    node_a = nodes[i]
    node_b = nodes[j]
    
    meta_a = node_meta[node_a]
    meta_b = node_meta[node_b]

    # Must be at the same physical station (sharing a gate NLC)
    common_gates = meta_a["Gate_NLCs"].intersection(meta_b["Gate_NLCs"])

    if common_gates:
      interchange_time = DEFAULT_INTERCHANGE_TIME
      source_type = "Default Rule (2.0 mins)"

      # Optional: Synthetic API Triad calculation for precision where NaPTANs exist
      naptan_a = meta_a["NaPTAN"]
      naptan_b = meta_b["NaPTAN"]

      if naptan_a and naptan_b and naptan_a != "nan" and naptan_b != "nan" and naptan_a != naptan_b:
        jp_url = f"https://api.tfl.gov.uk/Journey/JourneyResults/{urllib.parse.quote(naptan_a)}/to/{urllib.parse.quote(naptan_b)}"
        try:
          jp_res = requests.get(jp_url, params=params, timeout=3)
          if jp_res.status_code == 200:
            journeys = jp_res.json().get("journeys", [])
            if journeys:
              journey = journeys[0]
              total_journey_duration = float(journey.get("duration", 0))
              legs = journey.get("legs", [])
              
              if len(legs) >= 1:
                # Sum leg durations and wait times across legs
                sum_legs = sum(float(l.get("duration", 0)) for l in legs)
                sum_wait = sum(float(l.get("assessment", {}).get("averageWaitMinutes", 2.0)) for l in legs)
                
                # Synthetic Triad Formula: Walk = Journey - (Legs + Wait)
                calculated_walk = total_journey_duration - (sum_legs + sum_wait)
                if 0.5 <= calculated_walk <= 15.0:  # Reasonable walking transfer bounds
                  interchange_time = round(calculated_walk, 1)
                  source_type = "Synthetic API Triad"
        except Exception:
          pass

      # Add bidirectional interchange edges
      for f_node, t_node in [(node_a, node_b), (node_b, node_a)]:
        interchanges.append({
            "From_Node_ID": f_node,
            "To_Node_ID": t_node,
            "Shared_Gate_NLCs": str(list(common_gates)),
            "Interchange_Time_Mins": interchange_time,
            "Interchange_Type": "Behind Barrier",
            "Calculation_Method": source_type
        })

      time.sleep(0.01)  # Rate limit safety pause

# 4. Convert to Spark DataFrame and save to Delta table
pdf_interchanges = pd.DataFrame(interchanges)
interchange_times = spark.createDataFrame(pdf_interchanges)

interchange_times.write.mode("overwrite").saveAsTable("adlsg2analyticsprd001_sandbox_payments.ref.interchange_times")
print(f"Successfully generated and saved {interchange_times.count():,} directional behind-the-barrier interchange edges to a table.")

display(interchange_times)

Evaluating behind-the-barrier interchanges across 1242 nodes...
Successfully generated and saved 1,214 directional behind-the-barrier interchange edges to a table.


From_Node_ID,To_Node_ID,Shared_Gate_NLCs,Interchange_Time_Mins,Interchange_Type,Calculation_Method
Abbey Wood_elizabeth_5131,Abbey Wood_thameslink_5131,[5131],2.0,Behind Barrier,Default Rule (2.0 mins)
Abbey Wood_thameslink_5131,Abbey Wood_elizabeth_5131,[5131],2.0,Behind Barrier,Default Rule (2.0 mins)
Abbey Wood_elizabeth_5131,Abbey Wood_southeastern_5131,[5131],2.0,Behind Barrier,Default Rule (2.0 mins)
Abbey Wood_southeastern_5131,Abbey Wood_elizabeth_5131,[5131],2.0,Behind Barrier,Default Rule (2.0 mins)
Abbey Wood_thameslink_5131,Abbey Wood_southeastern_5131,[5131],2.0,Behind Barrier,Default Rule (2.0 mins)
Abbey Wood_southeastern_5131,Abbey Wood_thameslink_5131,[5131],2.0,Behind Barrier,Default Rule (2.0 mins)
Acton Main Line_elizabeth_3000,Acton Main Line_great-western-railway_3000,[3000],2.0,Behind Barrier,Default Rule (2.0 mins)
Acton Main Line_great-western-railway_3000,Acton Main Line_elizabeth_3000,[3000],2.0,Behind Barrier,Default Rule (2.0 mins)
Acton Town_district_500,Acton Town_piccadilly_500,[500],2.0,Behind Barrier,Default Rule (2.0 mins)
Acton Town_piccadilly_500,Acton Town_district_500,[500],2.0,Behind Barrier,Default Rule (2.0 mins)


## Mapping the Network

Constructs a directed graph representing the London contactless transit network:

- **NLC Aliasing**: Harmonises variant, and adjacent station codes (e.g., Monument Bank, Bow Road Bow Church) using NLC_ALIASES to ensure methodological consistency.

- **Nodes**: Represent unique line-station combinations, keeping multi-modal platforms (London Bridge NR vs. Underground) separate.

- **Edges**: Journey travel times, boarding wait times, and in-station interchange walking times.

In [0]:
# ==============================================================================
# CONSTRUCT NETWORK 
# ==============================================================================
# Combines travel times, interchanges, wait times, and station reference nodes 
# into clean Delta tables ready for NetworkX pathfinding and routing analytics.
# ==============================================================================

from pyspark.sql import functions as F

# 1. Load core network components from Delta tables
travel_times_df = spark.table("adlsg2analyticsprd001_sandbox_payments.ref.adjacent_station_travel_times")
interchanges_df = spark.table("adlsg2analyticsprd001_sandbox_payments.ref.interchange_times")
ref_df = spark.table("adlsg2analyticsprd001_sandbox_payments.ref.station_reference")

# 2. Build Vertices
vertices_df = (
    ref_df.filter(~F.col("Line").isin(["tfl-rail", "london-overground"]))
    .withColumn(
        "id",
        F.concat_ws("_", F.col("Station"), F.col("Line"), F.col("Gate_NLC")),
    )
    .select(
        F.col("id"),
        F.col("Station").alias("station_name"),
        F.col("Line").alias("line"),
    )
    .dropDuplicates(["id"])
)

# 3. Define default mode/line wait lookup dictionary for routing weights
line_wait_lookup = {
    "victoria": 2.5,
    "jubilee": 2.5,
    "northern": 2.5,
    "central": 2.5,
    "bakerloo": 2.5,
    "piccadilly": 2.5,
    "district": 2.5,
    "circle": 2.5,
    "hammersmith-city": 2.5,
    "metropolitan": 2.5,
    "waterloo-city": 2.5,
    "dlr": 2.5,
    "elizabeth": 2.0,
    "overground": 5.0,
}

# 4. Build Edges in Pandas for NetworkX conversion (including wait time penalty on train runs)
pdf_train = travel_times_df.select(
    F.col("From_Node_ID").alias("src"),
    F.col("To_Node_ID").alias("dst"),
    F.col("Travel_Time_Mins").cast("double").alias("run_time"),
    F.col("Line").alias("line"),
).toPandas()

# Add default wait time to train travel time (Total Edge Cost = Run Time + Wait Time)
pdf_train["default_wait"] = pdf_train["line"].apply(
    lambda l: line_wait_lookup.get(str(l).lower().strip(), 3.0)
)
pdf_train["weight"] = pdf_train["run_time"] + pdf_train["default_wait"]
pdf_train["edge_type"] = "Train Run"

pdf_interchange = interchanges_df.select(
    F.col("From_Node_ID").alias("src"),
    F.col("To_Node_ID").alias("dst"),
    F.col("Interchange_Time_Mins").cast("double").alias("weight"),
    F.lit("Walk/Transfer").alias("line"),
    F.lit("Interchange").alias("edge_type"),
).toPandas()

# Combine edges
pdf_edges = pd.concat([pdf_train, pdf_interchange], ignore_index=True)

# 5. Build NetworkX DiGraph
G = px_net.from_pandas_edgelist(
    pdf_edges,
    source="src",
    target="dst",
    edge_attr=["weight", "edge_type", "line"],
    create_using=px_net.DiGraph(),
)

print(
    f"Successfully built clean NetworkX graph: {G.number_of_nodes():,} nodes,"
    f" {G.number_of_edges():,} edges (Wait times factored into edge weights)."
)

Successfully built clean NetworkX graph: 1,056 nodes, 3,018 edges (Wait times factored into edge weights).


**todo**: visualise the network graph 

### DLR Interchanges
Executes Dijkstra shortest-path routing across the network graph from all non-DLR station gate lines to all DLR station endpoints (and vice versa):

DLR Interchange Identification: Traces each shortest path to identify the exact node where the journey transitions onto or off the DLR network.

Output Schema: Produces a comprehensive CSV table (dlr_journey_interchanges.csv) containing origin/destination names, NLCs, designated DLR interchanges, full interchange station lists, and formatted journey times.

In [0]:
# ==============================================================================
# DLR INTERCHANGE
# ==============================================================================

# Stations that count as DLR-eligible (45 core DLR stations + specified major interchange hubs)
DLR_ELIGIBLE_STATIONS = {
    # Hubs / Interchanges specified
    "bank",
    "west ham",
    "canning town",
    "stratford",
    "lewisham",
    "bow road",
    "monument",
    "tower hill",
    "greenwich",
    "woolwich arsenal",
}


def run_batch_dlr_routing(G, df_stations, nlc_to_name, NLC_ALIASES={}):
  """Runs batch routing between DLR-eligible nodes and Non-DLR nodes across the graph."""

  # 1. Classify all graph nodes into DLR vs Non-DLR based on station name or line
  dlr_nodes = set()
  non_dlr_nodes = set()

  for node in G.nodes:
    parts = str(node).split("_")
    # Assuming Node_ID format: Station_Line_Gate_NLC or Station_Line_NLC
    station_name = parts[0].strip().lower()
    line_name = parts[1].strip().lower() if len(parts) > 1 else ""

    is_dlr_line = "dlr" in line_name
    is_designated_hub = station_name in DLR_ELIGIBLE_STATIONS

    if is_dlr_line or is_designated_hub:
      dlr_nodes.add(node)
    else:
      non_dlr_nodes.add(node)

  print(
      f"Node Classification -> DLR Nodes: {len(dlr_nodes):,}, Non-DLR Nodes:"
      f" {len(non_dlr_nodes):,}"
  )

  results = []

  def get_route_interchanges_and_dlr_point(path):
    dlr_interchange_nlc = None
    dlr_interchange_name = None
    all_interchanges = []

    for i in range(len(path) - 1):
      u_node, v_node = path[i], path[i + 1]
      u_parts = u_node.split("_")
      v_parts = v_node.split("_")

      u_line = u_parts[1].lower() if len(u_parts) > 1 else ""
      v_line = v_parts[1].lower() if len(v_parts) > 1 else ""
      u_stat = u_parts[0]
      v_stat = v_parts[0]

      # Detect line transition (Interchange at same station)
      if u_line != v_line and u_stat == v_stat:
        if u_stat not in all_interchanges:
          all_interchanges.append(u_stat.title())

      # Detect DLR transition
      if ("dlr" in u_line and "dlr" not in v_line) or (
          "dlr" not in u_line and "dlr" in v_line
      ):
        trans_stat = u_stat if "dlr" in u_line else v_stat
        dlr_interchange_name = trans_stat.title()

    all_interchanges_str = (
        ", ".join(all_interchanges) if all_interchanges else "Direct / None"
    )
    return dlr_interchange_name, all_interchanges_str

  def add_route_record(direction, s_node, t_node, cost, path):
    total_mins = int(round(cost))
    hours, mins = divmod(total_mins, 60)
    journey_time_str = f"{hours}:{mins:02d}"

    s_parts = s_node.split("_")
    t_parts = t_node.split("_")
    from_station = s_parts[0].title()
    to_station = t_parts[0].title()

    ic_name, all_ics = get_route_interchanges_and_dlr_point(path)

    results.append({
        "from station": from_station,
        "to station": to_station,
        "dlr interchange": ic_name or "N/A",
        "all interchange stations": all_ics,
        "journey time": journey_time_str,
        "Journey_Time_Mins": total_mins,
        "Direction": direction,
    })

  # 2. Iterate across unique station pairs between Non-DLR and DLR nodes
  # Group nodes by station name to avoid redundant intra-station pairings
  from collections import defaultdict

  non_dlr_by_station = defaultdict(list)
  for n in non_dlr_nodes:
    non_dlr_by_station[n.split("_")[0].lower()].append(n)

  dlr_by_station = defaultdict(list)
  for n in dlr_nodes:
    dlr_by_station[n.split("_")[0].lower()].append(n)

  print(
      f"Routing across {len(non_dlr_by_station)} Non-DLR stations and"
      f" {len(dlr_by_station)} DLR stations..."
  )

  # Non-DLR -> DLR
  for s_stat, s_nodes in non_dlr_by_station.items():
    for t_stat, t_nodes in dlr_by_station.items():
      if s_stat == t_stat:
        continue

      best_cost, best_path = float("inf"), None
      for s in s_nodes:
        for t in t_nodes:
          if nx.has_path(G, s, t):
            c = nx.shortest_path_length(G, s, t, weight="weight")
            if c < best_cost:
              best_cost, best_path = c, nx.shortest_path(
                  G, s, t, weight="weight"
              )

      if best_path:
        add_route_record("Non-DLR to DLR", s_nodes[0], t_nodes[0], best_cost, best_path)

  df_out = pd.DataFrame(results).drop_duplicates(
      subset=["from station", "to station"]
  )
  print(
      "Routing Complete: Successfully generated"
      f" {len(df_out):,} route combinations."
  )
  return df_out

display(route_dlr)

### Manual vs Network DLR Interchanges

Compare existing and new interchange result files.

There are
- 1,242 unique nodes
- 45 DLR nodes, 23 adjesent nodes (eg. Bow Road-district-528, Stratford-elizabeth-719, Woolwich Arsenal-thameslink-5152), in total 68 nodes
- Making the other 1,174 stations non-DLR

There are 79,832 possible route interchnages (68 * 1174 = 79832).


In [0]:
spark.sql("use catalog syn_ldw_prd001_syn_ddctd_ldw")
spark.sql("use schema sbx_payments")

# Define reference NLC dataframe
dlr_manual_interchange = spark.sql("""
    SELECT *
    FROM ref_dlr_int
""")

display(dlr_manual_interchange)

NLC1,NLC2,DLR_INT
513,5057,940
5146,860,940
523,5403,513
528,6093,513
533,6008,513
5364,538,884
538,759,513
5405,883,884
5431,528,719
550,842,513


**todo**: graphs for comparison